# BFF Camera Demo - Training notebook (Google Colab)

Trains the YOLOX-style bucket detector + 5-class fill classifier against
the labeled dataset in this Colab session, exports ONNX, and (optionally)
uploads the artifacts to Supabase Storage `models-public/`.

Usage:
1. Open in Colab (free tier is enough for 1-2 epoch smoke).
2. Fill the cell below with credentials (or rely on the BFF Settings UI
   that this notebook writes to).
3. Run all cells.

In [ ]:
# 0. Environment setup - clone the repo, install deps.
# Colab Secrets: REPO_URL (e.g. https://github.com/<org>/<repo>.git) and,
# for a private repo, GITHUB_TOKEN (scope: repo).
from google.colab import userdata

REPO_URL = userdata.get('REPO_URL')
try:
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = ''
CLONE_URL = (REPO_URL if not TOKEN
             else REPO_URL.replace('https://', f'https://x-access-token:{TOKEN}@'))
!git clone {CLONE_URL} /content/camera-ai 2>/dev/null || (cd /content/camera-ai && git pull)
# Order matters: the `ml` package needs bff-backend, which is not on PyPI.
%pip install -e /content/camera-ai/backend[cuda,cpu]
%pip install -e /content/camera-ai/ml


In [ ]:
# 1. Credentials - filled in by the user.
import os
from getpass import getpass

os.environ['SUPABASE_URL'] = os.environ.get('SUPABASE_URL', 'https://naltduhojaazpodznvry.supabase.co')
os.environ['SUPABASE_SERVICE_ROLE_KEY'] = getpass('Supabase service role key: ')
os.environ['S3_ENDPOINT_URL'] = os.environ.get('S3_ENDPOINT_URL', 'https://naltduhojaazpodznvry.supabase.co/storage/v1/s3')
os.environ['S3_REGION'] = os.environ.get('S3_REGION', 'ap-southeast-1')
os.environ['S3_ACCESS_KEY_ID'] = getpass('S3 access key: ')
os.environ['S3_SECRET_ACCESS_KEY'] = getpass('S3 secret key: ')
os.environ['INFERENCE_PROVIDER'] = 'cuda'

In [ ]:
# 2. Verify the GPU is reachable.
import torch
print(f'torch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    !nvidia-smi

In [ ]:
# 3. Mount Drive (where the labeled dataset lives).
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/bff-data'  # adjust
import os; os.makedirs(DATA_DIR, exist_ok=True)
print(f'data dir: {DATA_DIR}')
print('contents:', os.listdir(DATA_DIR)[:10])

In [ ]:
# 4. Run training. Edit args as needed.
# DATA_DIR must point at a COCO root: annotations.json + images/.
%cd /content/camera-ai
!python -m ml train \
    --data-source=local \
    --data-path=$DATA_DIR/detector \
    --kind=detector \
    --model-name=bucket-detector \
    --version=0.1.0 \
    --epochs=50 \
    --batch-size=16 \
    --lr=1e-3 \
    --device=cuda \
    --no-sync


In [ ]:
# 4b. Fill classifier (5 classes: empty/partial/full/over/not_visible).
%cd /content/camera-ai
!python -m ml train \
    --data-source=local \
    --data-path=$DATA_DIR/classifier \
    --kind=classifier \
    --model-name=fill-classifier \
    --version=0.1.0 \
    --epochs=120 \
    --batch-size=16 \
    --lr=1e-3 \
    --device=cuda \
    --no-sync


In [ ]:
# 5. Inspect the latest run, then sync to Supabase if it looks good.
import os, glob
runs = sorted(glob.glob('/content/camera-ai/runs/*'), key=os.path.getmtime)
latest = runs[-1]
print('latest run:', latest)
print('files:', os.listdir(latest))
%cd /content/camera-ai
!python -m ml sync --run-dir={latest} --name=bucket-detector --version=0.1.0 --kind=detector


After the sync, the new weights are at `s3://models-public/bucket-detector/0.1.0/detector.onnx`
and the BE will pick them up on its next 5-minute refresh (or on the
next `/healthz` if you set `MODEL_REFRESH_INTERVAL_S=60`).